<h1 style="text-align: center;">Task-3P</h1>
<h3 style="text-align: right;">Name: Yatharth Deoly</h3>
<h3 style="text-align: right;">StudentId: 224207854</h3>
<h3 style="text-align: right;">EmailId: yatharthdeoly@gmail.com</h3>


## Introduction

In this task, we will be working with Real Estate Valuation dataset that has been provided by UCI. The real estate valuation is a regression problem. The market historical data set of real estate valuation are collected from Sindian Dist., New Taipei City, Taiwan.
Here we will be using ML regression models to predict the house price.

Explaining features and targets:

- FEATURES

  - No -> ID column
  - X1 transaction date -> for example, 2013.250=2013 March, 2013.500=2013 June, etc.
  - X2 house age -> in years
  - X3 distance to the nearest MRT station -> in meter
  - X4 number of convenience stores -> number of convenience stores in the living circle on foot
  - X5 latitude -> geographic coordinate, latitude
  - X6 longitude -> geographic coordinate, longitude

- TARGET
  - Y house price of unit area -> 10000 New Taiwan Dollar/Ping, where Ping is a local unit, 1 Ping = 3.3 meter squared


Defining imports at the top that will be used by the report.
In this case, having all the imports at the top is more intuitive than having some of them at the top and some of them scattered over the file.


In [1]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, make_scorer
from sklearn.linear_model import LinearRegression, Ridge, Lasso



from sklearn.model_selection import train_test_split, KFold, GridSearchCV, LeaveOneOut, cross_val_score
from sklearn.preprocessing import LabelEncoder, StandardScaler
import pandas as pd
import numpy as np
from scipy import stats

import warnings

# Ignore all warnings generated by the Python program
warnings.filterwarnings("ignore")

# Initialize the label encoder
label_encoder = LabelEncoder()

# Apply min-max scaling
standard_scaler = StandardScaler()

# Creating a list that will be used for keeping all the metrics of different models
train_model_lists = []

# Initialize the randomness in our different models and areas
random_state = 42

# Custom scorers for MSE, RMSE, and R2
mse_scorer = make_scorer(mean_squared_error, greater_is_better=False)
rmse_scorer = make_scorer(lambda y, y_pred: np.sqrt(
    mean_squared_error(y, y_pred)), greater_is_better=False)
r2_scorer = make_scorer(r2_score)

Extracting data into pandas Dataframe from a file.


In [2]:
# Exclude the 'No' column from the dataset as it contains ID
estate_df = pd.read_excel(
    "Real estate valuation data set.xlsx", usecols=lambda x: x != 'No')

# Printing random 5 data from our dataframe
estate_df.sample(n=5, random_state=random_state)

,X1 transaction date,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area
358,2013.166667,1.1,193.5845,6,24.96571,121.54089,45.1
350,2013.000000,13.2,492.2313,5,24.96515,121.53737,42.3
373,2013.083333,0.0,274.0144,1,24.97480,121.53059,52.2
399,2012.916667,12.7,170.1289,1,24.97371,121.52984,37.3
369,2012.666667,20.2,2185.1280,3,24.96322,121.51237,22.8


In [3]:
# Check for missing values
estate_df.isnull().sum()

X1 transaction date                       0
X2 house age                              0
X3 distance to the nearest MRT station    0
X4 number of convenience stores           0
X5 latitude                               0
X6 longitude                              0
Y house price of unit area                0
dtype: int64

In [4]:
# Check for duplicates
estate_df.duplicated().sum()

0

As "X1 transaction date" feature is not having correct format for dates, so we will be rectifying that feature via theory as:

- Determine the fraction of the year that has passed (e.g., for March, it's roughly 3/12 = 0.250).
- Add this fraction to the base year to get the decimal representation.

So via this theory we will be making 2 features from this feature


In [5]:
# This function will be converting numeric part to months for better visualization
def generate_month(numeric_month):
    months = ["December", "January", "February", "March", "April", "May",
              "June", "July", "August", "September", "October", "November"]
    return months[int(numeric_month * 12) % 12]


# This function will be converting numeric part to year for better visualization
def generate_data(decimal_year):
    return int(decimal_year), generate_month(decimal_year - int(decimal_year))

In [6]:
estate_df[['Txn year', 'Txn month']] = estate_df['X1 transaction date'].apply(
    lambda x: pd.Series(generate_data(x)))

estate_df.drop("X1 transaction date", inplace=True, axis=1)

estate_df.head()

,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Y house price of unit area,Txn year,Txn month
0,32.0,84.87882,10,24.98298,121.54024,37.9,2012,November
1,19.5,306.59470,9,24.98034,121.53951,42.2,2012,November
2,13.3,561.98450,5,24.98746,121.54391,47.3,2013,June
3,13.3,561.98450,5,24.98746,121.54391,54.8,2013,June
4,5.0,390.56840,5,24.97937,121.54245,43.1,2012,September


Outliers can be detected using various methods such as the IQR method or Z-score and imputing using median.


In [7]:
# Detecting outliers using Z-score
z_scores = np.abs(stats.zscore(estate_df.select_dtypes(include=[np.number])))
outliers = (z_scores > 3).sum(axis=0)

outliers

X2 house age                              0
X3 distance to the nearest MRT station    5
X4 number of convenience stores           0
X5 latitude                               1
X6 longitude                              5
Y house price of unit area                1
Txn year                                  0
dtype: int64

As from the above outlier analysis:

- Featue `Y house price of unit area` contains 1 outlier, so currently we won't remove the outlier.
- Feature `X6 longitude`, `X5 latitude` contains logitudes and latitudes of houses, which will be type of facts and that not need to be changed.
- Feature `X3 distance to the nearest MRT station` contains the distance to the nearest MRT station, which will helpful in our predictions.

So from above analysis we will be using same features without any outliers imputation.


In [8]:
# Define features and target
X = estate_df.drop('Y house price of unit area', axis=1)
y = estate_df['Y house price of unit area']

# Label encoding for ordinal categorical features
X['Txn year'] = label_encoder.fit_transform(X['Txn year'].to_numpy())
X['Txn month'] = label_encoder.fit_transform(X['Txn month'].to_numpy())

# Fit and transform the features
X_scaled = standard_scaler.fit_transform(X)
X_scaled = pd.DataFrame(X_scaled, columns=X.columns)
X_scaled.head()

,X2 house age,X3 distance to the nearest MRT station,X4 number of convenience stores,X5 latitude,X6 longitude,Txn year,Txn month
0,1.255628,-0.792495,2.007407,1.125430,0.448762,-1.511858,1.081700
1,0.157086,-0.616612,1.667503,0.912444,0.401139,-1.511858,1.081700
2,-0.387791,-0.414015,0.307885,1.486860,0.688183,0.661438,-0.286397
3,-0.387791,-0.414015,0.307885,1.486860,0.688183,0.661438,-0.286397
4,-1.117223,-0.549997,0.307885,0.834188,0.592937,-1.511858,1.537732


In [9]:
# Split the dataset into training and testing sets (80% train, 20% test)
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=random_state)

print(f"X_train dataset have {X_train.shape[0]} entries with {X_train.shape[1]} features.")
print(f"X_test dataset have {X_test.shape[0]} entries with {X_test.shape[1]} features.")
print(f"Target y_train dataset have {y_train.shape[0]} entries.")
print(f"Target y_test dataset have {y_test.shape[0]} entries.")

X_train dataset have 331 entries with 7 features.
X_test dataset have 83 entries with 7 features.
Target y_train dataset have 331 entries.
Target y_test dataset have 83 entries.


Train a Linear Regression Model


In [10]:
# Train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
mae = round(mean_absolute_error(y_test, y_pred), 3)
mse = round(mean_squared_error(y_test, y_pred), 3)
rmse = round(np.sqrt(mean_squared_error(y_test, y_pred)), 3)
r2 = round(r2_score(y_test, y_pred), 3)
train_accuracy = round(model.score(X_train, y_train), 3)
test_accuracy = round(model.score(X_test, y_test), 3)

print(f'Train Accuracy: {train_accuracy}')
print(f'Test Accuracy: {test_accuracy}')
print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

train_model_lists.append(["Simple_Linear_Regression_Model", train_accuracy, test_accuracy, mae, mse, rmse, r2])

Train Accuracy: 0.553
Test Accuracy: 0.695
MAE: 5.135
MSE: 51.146
RMSE: 7.152
R2 Score: 0.695


Leave-One-Out Cross-Validation


In [11]:
loo = LeaveOneOut()
model_loo = LinearRegression()


mae_scores = cross_val_score(model_loo, X_scaled, y, cv=loo, scoring='neg_mean_absolute_error', n_jobs=-1)
mse_scores = cross_val_score(model_loo, X_scaled, y, cv=loo, scoring=mse_scorer, n_jobs=-1)
rmse_scores = cross_val_score(model_loo, X_scaled, y, cv=loo, scoring=rmse_scorer, n_jobs=-1)
r2_scores = cross_val_score(model_loo, X_scaled, y, cv=loo, scoring=r2_scorer, n_jobs=-1)

# Train accuracy for each fold
train_accuracies = []
test_accuracies = []

for train_index, test_index in loo.split(X_scaled):
    X_loo_train, X_loo_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_loo_train, y_loo_test = y.iloc[train_index], y.iloc[test_index]

    model_loo.fit(X_loo_train, y_loo_train)

    train_accuracies.append(model_loo.score(X_loo_train, y_loo_train))
    test_accuracies.append(model_loo.score(X_loo_test, y_loo_test))


# Evaluate the model
mae = round(-mae_scores.mean(), 3)
mse = round(-mse_scores.mean(), 3)
rmse = round(-rmse_scores.mean(), 3)
r2 = round(r2_scores.mean(), 3)
train_accuracy = round(np.mean(train_accuracies), 3)
test_accuracy = round(np.mean(test_accuracies), 3)

print(f'Train Accuracy: {train_accuracy}')
print(f'Test Accuracy: {test_accuracy}')
print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

train_model_lists.append(["LeaveOneOut_Linear_Regression_Model", train_accuracy, test_accuracy, mae, mse, rmse, r2])

Train Accuracy: 0.581
Test Accuracy: nan
MAE: 6.233
MSE: 80.24
RMSE: 6.233
R2 Score: nan


Five-Fold Cross-Validation

In [12]:
kf = KFold(n_splits=5, shuffle=True, random_state=42)
model_kf = LinearRegression()

# Perform 5-Fold CV and get scores
mae_scores = cross_val_score(model_kf, X_scaled, y, cv=kf, scoring='neg_mean_absolute_error', n_jobs=-1)
mse_scores = cross_val_score(model_kf, X_scaled, y, cv=kf, scoring=mse_scorer, n_jobs=-1)
rmse_scores = cross_val_score(model_kf, X_scaled, y, cv=kf, scoring=rmse_scorer, n_jobs=-1)
r2_scores = cross_val_score(model_kf, X_scaled, y, cv=kf, scoring=r2_scorer, n_jobs=-1)

# Train accuracy for each fold
train_accuracies = []
test_accuracies = []

for train_index, test_index in kf.split(X_scaled):
    X_kf_train, X_kf_test = X_scaled.iloc[train_index], X_scaled.iloc[test_index]
    y_kf_train, y_kf_test = y.iloc[train_index], y.iloc[test_index]

    model_kf.fit(X_kf_train, y_kf_train)

    train_accuracies.append(model_kf.score(X_kf_train, y_kf_train))
    test_accuracies.append(model_kf.score(X_kf_test, y_kf_test))


# Evaluate the model
mae = round(-mae_scores.mean(), 3)
mse = round(-mse_scores.mean(), 3)
rmse = round(-rmse_scores.mean(), 3)
r2 = round(r2_scores.mean(), 3)
train_accuracy = round(np.mean(train_accuracies), 3)
test_accuracy = round(np.mean(test_accuracies), 3)

print(f'Train Accuracy: {train_accuracy}')
print(f'Test Accuracy: {test_accuracy}')
print(f'MAE: {mae}')
print(f'MSE: {mse}')
print(f'RMSE: {rmse}')
print(f'R2 Score: {r2}')

train_model_lists.append(["KFold(5)_Linear_Regression_Model", train_accuracy, test_accuracy, mae, mse, rmse, r2])

Train Accuracy: 0.583
Test Accuracy: 0.564
MAE: 6.265
MSE: 80.325
RMSE: 8.848
R2 Score: 0.564


L1 Regularization (Lasso) and L2 Regularization (Ridge) using GridSearchCV


In [13]:
# Defining model details and hyperparameters that will be used for Lasso and Ridge models.
model_details = {
    'Ridge_Regression_Model': Ridge(random_state=42),
    'Lassoo_Regression_Model': Lasso(random_state=42)
}
param_details = {
    'Ridge_Regression_Model': {'alpha': np.logspace(-8, 8, 100), 'fit_intercept': [True, False], 'solver': ['svd', 'cholesky', 'lsqr', 'sparse_cg', 'sag', 'saga', 'lbfgs'], 'tol': [1e-3, 1e-4, 1e-5, 1e-6]},
    'Lassoo_Regression_Model': {'alpha': np.logspace(-8, 8, 100), 'fit_intercept': [True, False], 'max_iter': [1000, 5000, 10000], 'tol': [1e-3, 1e-4, 1e-5, 1e-6], 'selection': ['cyclic', 'random']}
}

In [14]:
for key in model_details.keys():
    print("Running GridSearchCV for %s." % key)

    grid_search = GridSearchCV(model_details.get(key), param_details.get(key), cv=5, n_jobs=-1)
    grid_search.fit(X_train, y_train)

    print(f'Best hyperparameters that can be used for {key} => {grid_search.best_params_}')
    predicted_model = grid_search.best_estimator_

    predicted_model.fit(X_train, y_train)

    y_pred = predicted_model.predict(X_test)
    y_train_pred = predicted_model.predict(X_train)

    train_model_lists.append([key, round(predicted_model.score(X_train, y_train), 3), round(predicted_model.score(X_test, y_test), 3), round(mean_absolute_error(y_test, y_pred), 3), 
                              round(mean_squared_error(y_test, y_pred), 3), round(np.sqrt(mean_squared_error(y_test, y_pred)), 3), round(r2_score(y_test, y_pred), 3)])

    print("GridSearchCV for %s completed.\n" % key)

Running GridSearchCV for Ridge_Regression_Model.
Best hyperparameters that can be used for Ridge_Regression_Model => {'alpha': 16.297508346206467, 'fit_intercept': True, 'solver': 'saga', 'tol': 0.001}
GridSearchCV for Ridge_Regression_Model completed.

Running GridSearchCV for Lassoo_Regression_Model.
Best hyperparameters that can be used for Lassoo_Regression_Model => {'alpha': 0.27185882427329455, 'fit_intercept': True, 'max_iter': 1000, 'selection': 'random', 'tol': 0.001}
GridSearchCV for Lassoo_Regression_Model completed.



In [15]:
pd.DataFrame(train_model_lists, columns=['Model_Name', 'Train_Accuracy', 'Test_Accuracy',
             'MAE', 'MSE', 'RMSE', 'R2_Score']).sort_values(by=['R2_Score'], ascending=False)

,Model_Name,Train_Accuracy,Test_Accuracy,MAE,MSE,RMSE,R2_Score
3,Ridge_Regression_Model,0.552,0.697,5.158,50.788,7.127,0.697
0,Simple_Linear_Regression_Model,0.553,0.695,5.135,51.146,7.152,0.695
4,Lassoo_Regression_Model,0.551,0.693,5.226,51.448,7.173,0.693
2,KFold(5)_Linear_Regression_Model,0.583,0.564,6.265,80.325,8.848,0.564
1,LeaveOneOut_Linear_Regression_Model,0.581,NaN,6.233,80.240,6.233,NaN


Keyfindings for model analysis:
- Accuracy
    - The Ridge Regression Model and Simple Linear Regression Model have the highest test accuracy (0.697 and 0.695, respectively), showing better performance on unseen data compared to other models.

- Mean Absolute Error (MAE)
    - The Simple Linear Regression Model has the lowest MAE (5.135), suggesting it has the smallest average error magnitude on the test set.
    - Lassoo Regression Model has the highest MAE (5.226), indicating it may be less accurate compared to the other models.

- Mean Squared Error (MSE)
    - The Ridge Regression Model shows the lowest MSE (50.788), implying it has the smallest average squared error.
    - KFold (5) and LeaveOneOut models have significantly higher MSE values (80.325 and 80.240, respectively), indicating poorer performance.

- Root Mean Squared Error (RMSE)
    - The Ridge Regression Model has the lowest RMSE (7.127), which correlates with its low MSE, suggesting it has the least amount of error in terms of standard deviation.
    - KFold (5) model has the highest RMSE (8.848), indicating higher deviation in prediction errors.

- R<sup>2</sup> Score:
    - The Ridge Regression Model and Simple Linear Regression Model have the highest R<sup>2</sup> scores (0.697 and 0.695, respectively), indicating they explain a larger proportion of the variance in the target variable.
    - KFold (5) and LeaveOneOut models have lower R<sup>2</sup> scores (0.564 and NaN), suggesting they explain less variance and might be less effective.


Conclusion:
- Ridge Regression Model stands out with the best overall performance in terms of accuracy, MAE, MSE, RMSE, and R<sup>2</sup> score.
- Simple Linear Regression Model is comparable to Ridge Regression but slightly worse in some metrics.
- Lassoo Regression Model performs slightly worse than the Ridge and Simple Linear Regression Models.
- KFold (5) and LeaveOneOut models show lower performance metrics overall, especially in terms of accuracy, error metrics, and R<sup>2</sup> score, which suggests that these cross-validation techniques might not be as effective for this dataset or that there may be issues with the data split.


Recommendation: Based on the metrics, the Ridge Regression Model seems to be the best choice for this dataset, offering a balance of good accuracy and low error metrics.

# Summary

In this report we have learnt about different models i.e. Linear Regreassion, L1 Regularization (Lasso) and L2 Regularization (Ridge) with different validation like Leave-One-Out Cross-Validation and Five-Fold Cross-Validation using GridSearchCV. Here we have learnt about different libraries from python, sklearn from loading the file to the modelling and then checking metrics of different models.

# References

- https://archive.ics.uci.edu/dataset/477/real+estate+valuation+data+set
- Different blogs from https://medium.com/ like
  - https://medium.com/analytics-vidhya/understanding-the-linear-regression-808c1f6941c0
  - https://medium.com/@analyttica/understanding-lasso-regression-c8ffc2b1152f
  - https://medium.com/@abelkuriakose/ridge-regression-98c2a65cb3b1
- https://scikit-learn.org/stable/api/sklearn.preprocessing.html
- https://scikit-learn.org/stable/api/sklearn.linear_model.html
